# 4D SfM — DEM + DoD

Produces DEMs, orthos, and DoD rasters from the 4D SfM clouds.
Two interchangeable implementations — pick one:

- **Option A** — `cntp.io.build_dem_and_ortho` (scipy cubic griddata,
  single-threaded, slow but produces DEM + orthoimage).
- **Option B** — ASP `point2dem` wrapper (multi-threaded, fast, DEM
  only — no ortho).

Reference DEM/ortho are cached in `_ref_cache/`; re-runs skip the
rebuild unless `overwrite=True`.

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"

from pathlib import Path

import Metashape  # noqa: F401  — must import after AGISOFT_LICENSE_PATH is set
from cntp.pipeline_4dsfm import run_4dsfm_day

## Configuration

Edit only this section per-day. Everything below derives paths from these variables.

In [2]:
# ── Paths ────────────────────────────────────────────────────────────
base_dir     = Path("/mnt/g/2023_11_Nepal/2023_Changri")
tlcam_dir    = base_dir / "TLCAM"
output_dir   = base_dir

ref_cloud    = base_dir / "Ref_PC" / "Reference_UAV_TLC_PCS.laz"
glacier_mask = base_dir / "glaciermask_new" / "glacier_mask_pcs.shp"
registry_csv = output_dir / "output_new" / "reference_registry.csv"

# ── Date to process ──────────────────────────────────────────────────
new_date = "2023-12-30"

# ── Pipeline parameters (defaults shown — override as needed) ────────
params = dict(
    match_downscale = 0,
    depth_downscale = 2,
    loc_acc_new     = (0.5, 0.5, 0.5),
    rot_acc_new     = (5.0, 5.0, 5.0),
    ref_downsample  = 0.4,
    tba_downsample  = 1.0,
    p2p_max_disp    = 10.0,
    sp2p_max_disp   =  5.0,
    m_sp2p_max_disp =  0.5,
    use_ecef        = True,
    overwrite       = False,
    verbose         = True,
)

## Run pipeline (Steps 1–7)

Each step inside `run_4dsfm_day` skips itself if its key output already
exists. Flip `overwrite=True` in `params` above to force a full re-run.

In [3]:
result = run_4dsfm_day(
    new_date     = new_date,
    tlcam_dir    = tlcam_dir,
    ref_cloud    = ref_cloud,
    glacier_mask = glacier_mask,
    registry_csv = registry_csv,
    output_dir   = output_dir,
    **params,
)

print()
print(f"Coreg M3C2  : before {result['coreg_med_before']:+.4f} m  →  after {result['coreg_med_after']:+.4f} m")
print(f"Validation  : median {result['validation_med']:+.4f} m   std {result['validation_std']:.4f} m")

[Step 1] Skipping — 2023-12-30_cameras_4DSfM.csv exists
[Step 2] Skipping — 2023-12-30_cloud.las exists
  Stable reference cached → Reference_UAV_TLC_PCS_ds0.40_stable.las
[Step 3] Skipping — 2023-12-30_cloud_coreg_hsfm.las exists
[Step 3b] Skipping — stable TBA exists
[Step 4] Skipping — 2023-12-30_cameras_coreg.csv exists
[Step 6] Skipping — 2023-12-30_cloud_validated.laz exists
[Step 6b] Skipping — 2023-12-30_cloud_validated_stable.laz exists
[Step 7] Skipping — 2023-12-30 already in registry

Coreg M3C2  : before +nan m  →  after +nan m
Validation  : median +nan m   std nan m


## DEM + DoD

Two equivalent paths below. Run only one — both ultimately produce a
`DOD.tif` you can analyse with `plot_dod_histogram`.

### Option A — scipy cubic griddata (single-threaded)

Produces both DEM and orthoimage. RAM-heavy on dense clouds — dial
`ref_cloud_downsample` down if it OOMs.

In [4]:
import laspy
from cntp.raster import build_reference_dem_and_ortho

# ── Knobs ──────────────────────────────────────────────────────────
res                  = 1.0
ref_cloud_downsample = 0.25    # multiplies on top of params['ref_downsample']
overwrite_ref_dem    = False   # True ⇒ rebuild even if cached

# ── Paths derived from config (cell 3) ────────────────────────────
ref_cache_dir = output_dir / "output_new" / "_ref_cache"
ref_ds_path   = ref_cache_dir / f"{ref_cloud.stem}_ds{params['ref_downsample']:.2f}.las"
ref_input     = ref_ds_path if ref_ds_path.exists() else ref_cloud

with laspy.open(ref_cloud) as _f:
    utm_epsg = _f.header.parse_crs().to_epsg()

# ── Build reference DEM + ortho (cached in _ref_cache/) ───────────
ref_dem, ref_ortho = build_reference_dem_and_ortho(
    ref_cloud_path   = ref_input,
    cache_dir        = ref_cache_dir,
    res              = res,
    max_gap_pixels   = 1,
    utm_epsg         = utm_epsg,
    cloud_downsample = ref_cloud_downsample,
    overwrite        = overwrite_ref_dem,
)
print(f"Ref DEM   : {ref_dem}")
print(f"Ref Ortho : {ref_ortho}")

  DEM + ortho cached → reference_dem.tif, reference_ortho.tif
Ref DEM   : /mnt/g/2023_11_Nepal/2023_Changri/output_new/_ref_cache/reference_dem.tif
Ref Ortho : /mnt/g/2023_11_Nepal/2023_Changri/output_new/_ref_cache/reference_ortho.tif


In [5]:
from cntp.raster import build_dem_and_ortho

# ── Knobs ──────────────────────────────────────────────────────────
overwrite_day_dem = False    # True ⇒ rebuild even if cached

# ── Paths derived from config (cell 3) ────────────────────────────
day_dir     = output_dir / "output_new" / new_date
aligned_las = day_dir / "coreg" / f"{new_date}_cloud_coreg_hsfm.las"
single_day  = day_dir / "single_day"

# ── Build per-day DEM + ortho ─────────────────────────────────────
dem, ortho = build_dem_and_ortho(
    cloud_las        = aligned_las,
    ref_las          = ref_cloud,
    out_dir          = single_day,
    name_stem        = new_date,
    res              = res,
    max_gap_pixels   = 1,
    utm_epsg         = utm_epsg,
    cloud_downsample = params['tba_downsample'],
    overwrite        = overwrite_day_dem,
)
print(f"Day DEM   : {dem}")
print(f"Day Ortho : {ortho}")

  DEM + ortho cached → 2023-12-30_dem.tif, 2023-12-30_ortho.tif
Day DEM   : /mnt/g/2023_11_Nepal/2023_Changri/output_new/2023-12-30/single_day/2023-12-30_dem.tif
Day Ortho : /mnt/g/2023_11_Nepal/2023_Changri/output_new/2023-12-30/single_day/2023-12-30_ortho.tif


In [6]:
import rasterio
from cntp.raster import build_dod
from cntp.plot import plot_dod_histogram

overwrite_dod = False

dod_path = build_dod(
    ref_dem_path = ref_dem,
    day_dem_path = dem,
    out_path     = single_day / "DOD.tif",
    overwrite    = overwrite_dod,
)

with rasterio.open(dod_path) as src:
    dod_values = src.read(1)

stats = plot_dod_histogram(
    dod_values,
    output_dir = single_day,
    title      = f"scipy DoD — {new_date}",
    filename   = "dod_histogram.png",
)
print(stats)

  DoD cached → DOD.tif
{'median': -0.14205577272241499, 'mean': 0.0753838359665568, 'std': 7.092161837061735, 'n': 190372}


#### Stable-terrain DoD

Raster analogue of the point-cloud stable-terrain pipeline: apply
slope > 60° + NDWI/intensity (from the ortho) + glacier polygon
mask to both DEMs, then difference. Median should sit at ~0 m if
coreg is good — that's the QC check; the spread is the residual
noise floor on terrain that isn't supposed to be changing.

In [7]:
from cntp.raster import extract_stable_terrain_from_dem

# ── Knobs ──────────────────────────────────────────────────────────
slope_threshold         = 60.0
overwrite_stable        = False    # True ⇒ rebuild stable DEMs
overwrite_stable_dod    = False    # True ⇒ rebuild stable DoD

ref_ortho_path = ref_cache_dir / "reference_ortho.tif"
day_ortho_path = single_day    / f"{new_date}_ortho.tif"

# ── Stable reference DEM (cached in _ref_cache/) ─────────────────
ref_stable_dem = extract_stable_terrain_from_dem(
    dem_path          = ref_dem,
    ortho_path        = ref_ortho_path,
    glacier_mask_path = glacier_mask,
    slope_threshold   = slope_threshold,
    overwrite         = overwrite_stable,
)

# ── Stable day DEM ───────────────────────────────────────────────
day_stable_dem = extract_stable_terrain_from_dem(
    dem_path          = dem,
    ortho_path        = day_ortho_path,
    glacier_mask_path = glacier_mask,
    slope_threshold   = slope_threshold,
    overwrite         = overwrite_stable,
)

# ── Stable DoD ───────────────────────────────────────────────────
stable_dod_path = build_dod(
    ref_dem_path = ref_stable_dem,
    day_dem_path = day_stable_dem,
    out_path     = single_day / "DOD_stable.tif",
    overwrite    = overwrite_stable_dod,
)

# ── Histogram ────────────────────────────────────────────────────
with rasterio.open(stable_dod_path) as src:
    stable_dod_values = src.read(1)

stable_stats = plot_dod_histogram(
    stable_dod_values,
    output_dir = single_day,
    title      = f"scipy stable DoD — {new_date}",
    filename   = "dod_stable_histogram.png",
)
print(stable_stats)

  Stable DEM cached → reference_dem_stable.tif
  Stable DEM cached → 2023-12-30_dem_stable.tif
  DoD cached → DOD_stable.tif
{'median': -0.1705898803047603, 'mean': 0.17867455806571397, 'std': 6.416267055412727, 'n': 2129}


### M3C2 distance raster (alternative to vertical DoD)

Same idea as the DoD raster, but each pixel holds the **median
M3C2 distance** between the two clouds at that XY location rather
than `ref_z − day_z`. Because M3C2 measures perpendicular
cloud-to-cloud distance along the local surface normal, the result
is **immune to the `tan(slope)` projection** that inflates vertical
DoD on steep terrain.

Inputs: full day cloud + downsampled cached reference, no masking.
Aggregation: median per pixel.

M3C2 parameters (`normal_radii=2.5`, `cyl_radius=2.5`,
`max_distance=30`) match every other M3C2 call in the project —
Step 3b `evaluate_coreg` and Step 6b validation use the same numbers.

In [12]:
import laspy
import rasterio
from cntp.raster import m3c2_to_raster
from cntp.plot import plot_dod_histogram

# ── Knobs ──────────────────────────────────────────────────────────
res                 = 1.0
overwrite_m3c2      = False     # True ⇒ rebuild even if cached

# ── Paths derived from config (cell 3) ────────────────────────────
with laspy.open(ref_cloud) as _f:
    utm_epsg = _f.header.parse_crs().to_epsg()

ref_cache_dir = output_dir / "output_new" / "_ref_cache"
ref_cache_las = ref_cache_dir / f"{ref_cloud.stem}_ds{params['ref_downsample']:.2f}.las"
day_dir       = output_dir / "output_new" / new_date
aligned_las   = day_dir / "coreg" / f"{new_date}_cloud_coreg_hsfm.las"
single_day    = day_dir / "single_day"

# ── Build M3C2 raster ─────────────────────────────────────────────
# Grid is anchored to the original full-res ref bbox so the M3C2
# raster sits on the same grid as the scipy/ASP DEMs and DoDs —
# directly comparable cell-for-cell in QGIS.
m3c2_raster_path = m3c2_to_raster(
    ref_las         = ref_cache_las,
    day_las         = aligned_las,
    out_path        = single_day / "M3C2_raster.tif",
    grid_anchor_las = ref_cloud,
    res             = res,
    utm_epsg        = utm_epsg,
    ref_downsample  = 0.5,    # cache is already at params['ref_downsample']
    day_downsample  = 1.0,    # full day cloud
    overwrite       = overwrite_m3c2,
)

# ── Histogram ─────────────────────────────────────────────────────
with rasterio.open(m3c2_raster_path) as src:
    m3c2_values = src.read(1)

m3c2_stats = plot_dod_histogram(
    m3c2_values,
    output_dir = single_day,
    title      = f"M3C2 raster — {new_date}",
    filename   = "m3c2_raster_histogram.png",
)
print(m3c2_stats)

  Loading reference cloud (downsample=0.5) …
  Loading day cloud (downsample=1.0) …
  Ref pts : 25,222,994   |   Day pts : 6,661,824
  Running M3C2 (normal_radii=2.5 m, cyl_radius=2.5 m, max_distance=30.0 m) …
[2026-06-01 16:47:38][INFO] Building KDTree structure with leaf parameter 10
[2026-06-01 16:47:50][INFO] Building KDTree structure with leaf parameter 10
  M3C2 over all corepoints: median=+0.1118 m   std=1.3504 m
  Binning 8,233,473 valid corepoints into 1248×1307 grid (mean per cell) …
Saved: /mnt/g/2023_11_Nepal/2023_Changri/output_new/2023-12-30/single_day/M3C2_raster.tif
  M3C2 raster valid pixels : 231,570/1,631,136 (14.2%) → M3C2_raster.tif
{'median': 0.15192647004505472, 'mean': 0.1576350728586173, 'std': 1.530320762054169, 'n': 231570}


No response from license server lmw-2c.polimi.it:2020: Could not resolve hostname (6)
Can't recover floating license
No response from license server lmw-2c.polimi.it:2020: Could not resolve hostname (6)


### Option B — ASP `point2dem` (multi-threaded, fast)

Streaming binning, runs on all CPU cores. ~10–30× faster than scipy
cubic on large clouds; no overshoot at vertical features. Produces
DEM only (no orthoimage).

Three steps: (1) patch CRS into the cached downsampled LAS (one-time,
idempotent), (2) build both DEMs anchored to the reference grid,
(3) DoD + histogram.

In [ ]:
# One-time idempotent CRS patch: the cached _ref_cache/<stem>_ds<f>.las
# written by save_las before the CRS-preservation fix has no EPSG in its
# header — ASP defaults to lat/lon and produces an empty raster. This
# cell streams the file through laspy (low RAM) and injects the EPSG.
import shutil
import laspy
from pyproj import CRS

with laspy.open(ref_cloud) as _f:
    utm_epsg = _f.header.parse_crs().to_epsg()

ref_cache_dir = output_dir / "output_new" / "_ref_cache"
ref_cache_las = ref_cache_dir / f"{ref_cloud.stem}_ds{params['ref_downsample']:.2f}.las"

with laspy.open(ref_cache_las) as _f:
    _existing = _f.header.parse_crs()
if _existing is not None and _existing.to_epsg() == utm_epsg:
    print(f"{ref_cache_las.name} already tagged EPSG:{utm_epsg} — skipping.")
else:
    print(f"Patching CRS=EPSG:{utm_epsg} into {ref_cache_las.name} (streaming) …")
    tmp_path = ref_cache_las.with_suffix(".las.tmp")
    with laspy.open(ref_cache_las, mode="r") as src:
        src.header.add_crs(CRS.from_epsg(utm_epsg))
        with open(tmp_path, "wb") as _f:
            with laspy.LasWriter(_f, header=src.header) as writer:
                for chunk in src.chunk_iterator(500_000):
                    writer.write_points(chunk)
    shutil.move(str(tmp_path), str(ref_cache_las))
    print("Done.")

In [ ]:
from contributors.umayr.tools import point2dem

# ── Knobs ──────────────────────────────────────────────────────────
res                = 1.0
overwrite_asp_ref  = False    # True ⇒ rebuild reference DEM even if cached
overwrite_asp_day  = False    # True ⇒ rebuild day DEM even if cached

# ── Paths derived from config (cell 3) ────────────────────────────
asp_root      = output_dir / "output_new" / "ASP_output"
asp_ref_dir   = asp_root / "reference"
asp_day_dir   = asp_root / "single_day"
ref_cache_dir = output_dir / "output_new" / "_ref_cache"
ref_cache_las = ref_cache_dir / f"{ref_cloud.stem}_ds{params['ref_downsample']:.2f}.las"
aligned_las   = output_dir / "output_new" / new_date / "coreg" / f"{new_date}_cloud_coreg_hsfm.las"
single_day    = output_dir / "output_new" / new_date / "single_day"

# Both ASP DEMs anchored to the original reference cloud's bbox via
# --t_projwin, so they share grid (shape + transform) and the DoD
# below works without resampling.

# Reference DEM — skip if already cached.
ref_asp_dem = asp_ref_dir / "reference-DEM.tif"
if ref_asp_dem.exists() and not overwrite_asp_ref:
    print(f"Ref DEM cached → {ref_asp_dem}")
else:
    ref_asp_dem = point2dem(
        cloud_las  = ref_cache_las,
        out_prefix = asp_ref_dir / "reference",
        res        = res,
        utm_epsg   = utm_epsg,
        ref_las    = ref_cloud,
    )

# Day DEM — skip if already cached.
day_asp_dem = asp_day_dir / f"{new_date}-DEM.tif"
if day_asp_dem.exists() and not overwrite_asp_day:
    print(f"Day DEM cached → {day_asp_dem}")
else:
    day_asp_dem = point2dem(
        cloud_las  = aligned_las,
        out_prefix = asp_day_dir / new_date,
        res        = res,
        utm_epsg   = utm_epsg,
        ref_las    = ref_cloud,
    )

In [ ]:
import rasterio
from cntp.raster import build_dod
from cntp.plot import plot_dod_histogram

overwrite_asp_dod = False

asp_dod = build_dod(
    ref_dem_path = ref_asp_dem,
    day_dem_path = day_asp_dem,
    out_path     = asp_day_dir / "DOD.tif",
    overwrite    = overwrite_asp_dod,
)

with rasterio.open(asp_dod) as src:
    asp_dod_values = src.read(1)

stats = plot_dod_histogram(
    asp_dod_values,
    output_dir = asp_day_dir,
    title      = f"ASP DoD — {new_date}",
    filename   = "dod_histogram.png",
)
print(stats)